## Install & Import Dependencies

In [99]:
import pandas as pd
from rapidfuzz import process, fuzz
import re

In [100]:
# sample raw addresses
df = pd.read_excel("../../data/sample_address.xlsx")

# location reference (city, province, etc.)
dim_location = pd.read_csv("../../data/mapping/dim_location_20260316_v2.csv", encoding="cp1252")

alias = pd.read_csv("../../data/utils/ph_address_alias_extended_v4.csv", encoding="cp1252")

# Preview
df.head(), dim_location.head(), alias.head()

(                                order_deliveraddress
 0                             Tagumpay sindalan CSFP
 1  BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...
 2                                      Cainta Rizal 
 3                           2079 a elias st sta cruz
 4           Swani Hardware 77 tirso neri street CDOC,
    psgc_10_digit                             region_name province_name  \
 0     1400101001  Cordillera Administrative Region (CAR)          Abra   
 1     1400101002  Cordillera Administrative Region (CAR)          Abra   
 2     1400101003  Cordillera Administrative Region (CAR)          Abra   
 3     1400101004  Cordillera Administrative Region (CAR)          Abra   
 4     1400101005  Cordillera Administrative Region (CAR)          Abra   
 
   city_name barangay_name  region_code  province_code  city_code  \
 0  Bangued       Agtangao           14              1          1   
 1  Bangued          Angad           14              1          1   
 2  Bangued     

In [101]:
alias_map = (
    alias.assign(
        pattern=alias["pattern"].fillna("").str.lower().str.strip(),
        replacement=alias["replacement"].fillna("").str.lower().str.strip(),
    )
    .query("pattern != ''")
    .sort_values("pattern", key=lambda s: s.str.len(), ascending=False)
)

def normalize_address(address, alias_head=None):
    if alias_head is None:
        alias_head = alias_map
    if pd.isna(address):
        return ""

    text = str(address).lower()
    text = re.sub(r"[^\w\s]", " ", text)

    for pattern, replacement in alias_head[["pattern", "replacement"]].itertuples(index=False):
        text = re.sub(rf"\b{re.escape(pattern)}\b", replacement, text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

def rtl_check_address(normalized_add, dim_location, threshold=90):
    words = normalized_add.split()
    for end in range(len(words), 0, -1):
        candidate = " ".join(words[end-1:])
        match = process.extractOne(
            candidate,
            dim_location["city_name"].dropna().tolist(),
            scorer=fuzz.token_sort_ratio,
        )
        if match and match[1] >= threshold:
            return True, match[0], match[1]
    return False, None, None

df["normalized_add"] = df["order_deliveraddress"].apply(lambda x: normalize_address(x, alias_map))

In [102]:
def checking_city(normalized_add, dim_location):
    for city in dim_location["city_name"]:
        if pd.isna(city):
            continue
        
        city_clean = city.lower()
        
        if city_clean in normalized_add:
            return city
    
    return None

In [103]:
def fuzzy_matching(normalized_add, dim_location):
    
    matched_city = checking_city(normalized_add, dim_location)
    
    if matched_city:
        filtered_df = dim_location[
            dim_location["city_name"] == matched_city
        ]
    else:
        filtered_df = dim_location.copy()
    
    # Combine searchable fields
    filtered_df["full_location"] = (
        filtered_df["barangay_name"].fillna("") + " " +
        filtered_df["city_name"].fillna("") + " " +
        filtered_df["province_name"].fillna("")
    )
    
    choices = filtered_df["full_location"].tolist()
    
    match = process.extractOne(
        normalized_add,
        choices,
        scorer=fuzz.token_sort_ratio
    )
    
    if match:
        matched_text, score, idx = match
        return {
            "match": matched_text,
            "score": score,
            "city_name": filtered_df.iloc[idx]["city_name"],
            "barangay_name": filtered_df.iloc[idx]["barangay_name"],
            "province_name": filtered_df.iloc[idx]["province_name"]
        }
    
    return None

In [104]:
results = []
for _, row in df.iterrows():
    normalized_add = row["normalized_add"]

    rtl_ok, rtl_city, rtl_score = rtl_check_address(normalized_add, dim_location)
    if not rtl_ok:
        results.append(None)
        continue

    match_result = fuzzy_matching(normalized_add, dim_location)
    results.append(match_result)

df["match_result"] = results
df.head()

,order_deliveraddress,normalized_add,match_result
0,Tagumpay sindalan CSFP,tagumpay sindalan csfp,None
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",block 16 lot 4 quiotan street phil am life vil...,None
2,Cainta Rizal,cainta rizal,None
3,2079 a elias st sta cruz,2079 a elias street santa cruz,None
4,Swani Hardware 77 tirso neri street CDOC,swani hardware 77 tirso neri street cdoc,None


In [105]:
df["matched_city"] = df["match_result"].apply(lambda x: x["city_name"] if x else None)
df["matched_barangay"] = df["match_result"].apply(lambda x: x["barangay_name"] if x else None)
df["matched_province"] = df["match_result"].apply(lambda x: x["province_name"] if x else None)
df["match_score"] = df["match_result"].apply(lambda x: x["score"] if x else None)

df.head()

,order_deliveraddress,normalized_add,match_result,matched_city,matched_barangay,matched_province,match_score
0,Tagumpay sindalan CSFP,tagumpay sindalan csfp,None,NaN,NaN,NaN,NaN
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",block 16 lot 4 quiotan street phil am life vil...,None,NaN,NaN,NaN,NaN
2,Cainta Rizal,cainta rizal,None,NaN,NaN,NaN,NaN
3,2079 a elias st sta cruz,2079 a elias street santa cruz,None,NaN,NaN,NaN,NaN
4,Swani Hardware 77 tirso neri street CDOC,swani hardware 77 tirso neri street cdoc,None,NaN,NaN,NaN,NaN


In [106]:
THRESHOLD = 80

df["is_confident"] = df["match_score"] >= THRESHOLD
df.head()

,order_deliveraddress,normalized_add,match_result,matched_city,matched_barangay,matched_province,match_score,is_confident
0,Tagumpay sindalan CSFP,tagumpay sindalan csfp,None,NaN,NaN,NaN,NaN,False
1,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",block 16 lot 4 quiotan street phil am life vil...,None,NaN,NaN,NaN,NaN,False
2,Cainta Rizal,cainta rizal,None,NaN,NaN,NaN,NaN,False
3,2079 a elias st sta cruz,2079 a elias street santa cruz,None,NaN,NaN,NaN,NaN,False
4,Swani Hardware 77 tirso neri street CDOC,swani hardware 77 tirso neri street cdoc,None,NaN,NaN,NaN,NaN,False


In [ ]:

# Normalize reference columns
dim_location["city_clean"] = dim_location["city_name"].apply(normalize_address)
dim_location["prov_clean"] = dim_location["province_name"].apply(normalize_address)

# Merge reference info into main df if needed (example: assume you already have expected city/province)
# If not, you can skip merge and just use available columns

df["addr_clean"] = df["normalized_add"]

# Normalize fields on df (not only on dim_location)
df["addr_clean"] = df["normalized_add"].fillna("").apply(normalize_address)

# Use matched outputs from previous step
df["city_clean"] = df["matched_city"].fillna("").apply(normalize_address)
df["prov_clean"] = df["matched_province"].fillna("").apply(normalize_address)


KeyboardInterrupt: 

In [ ]:
df['city_match'] = df.apply(
    lambda x: x['city_clean'] in x['addr_clean'] if pd.notna(x['city_clean']) else False,
    axis=1
)

df['province_match'] = df.apply(
    lambda x: x['prov_clean'] in x['addr_clean'] if pd.notna(x['prov_clean']) else False,
    axis=1
)

In [ ]:
from rapidfuzz import fuzz

def fuzzy_match(a, b, threshold=80):
    if pd.isna(a) or pd.isna(b):
        return False
    return fuzz.partial_ratio(a, b) >= threshold

In [ ]:
df['city_match_fuzzy'] = df.apply(
    lambda x: fuzzy_match(x['city_clean'], x['addr_clean']),
    axis=1
)

df['province_match_fuzzy'] = df.apply(
    lambda x: fuzzy_match(x['prov_clean'], x['addr_clean']),
    axis=1
)

In [ ]:
df["city_match_final"] = df["city_match"] | df["city_match_fuzzy"]
df["province_match_final"] = df["province_match"] | df["province_match_fuzzy"]
df.to_excel("../../data/output/valid/matched_sample_addresses.xlsx", index=False)